In [1]:
!pip install transformers accelerate torch pandas pydantic


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import pandas as pd
import json
import re

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from pydantic import BaseModel

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Qwen Loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0


Qwen Loaded


In [4]:
ocr_documents_df = pd.DataFrame([
    {
        "customer_id":1,
        "document_text":"""
Name: Aarav Mehta
DOB: 12/06/1994
PAN: ABCDE1234F
"""
    }
])

ocr_documents_df

,customer_id,document_text
0,1,\nName: Aarav Mehta\nDOB: 12/06/1994\nPAN: ABC...


In [5]:
customer_master_df = pd.DataFrame([
    {
        "customer_id":1,
        "name":"Aarav Mehta",
        "dob":"12/06/1994",
        "pan":"ABCDE1234F",
        "income":1200000
    }
])

customer_master_df

,customer_id,name,dob,pan,income
0,1,Aarav Mehta,12/06/1994,ABCDE1234F,1200000


In [6]:
transactions_df = pd.DataFrame([
    {
        "customer_id":1,
        "amount":50000,
        "type":"credit",
        "mode":"salary"
    },

    {
        "customer_id":1,
        "amount":25000,
        "type":"debit",
        "mode":"cash"
    },

    {
        "customer_id":1,
        "amount":15000,
        "type":"debit",
        "mode":"cash"
    }
])

transactions_df

,customer_id,amount,type,mode
0,1,50000,credit,salary
1,1,25000,debit,cash
2,1,15000,debit,cash


In [7]:
#Pydantic models
from pydantic import BaseModel
from typing import List


class OCRResult(BaseModel):
    name:str
    dob:str
    pan:str


class IdentityResult(BaseModel):
    match_score:float
    match_status:str
    reasoning:str


class ComplianceResult(BaseModel):
    sanctions_match:bool
    pep_match:bool
    risk_level:str
    reasoning:str


class TransactionFeatures(BaseModel):
    total_credit:float
    total_debit:float
    cash_ratio:float
    avg_transaction:float
    high_value_txn_count:int


class FinancialProfile(BaseModel):
    profile:str
    risk_indicators:List[str]


class RiskResult(BaseModel):

    risk_score: float
    risk_level: str
    explanation: str


# class ExplainabilityResult(BaseModel):
#     reasons:List[str]


class HumanReviewResult(BaseModel):

    escalate: bool
    review_queue: str

In [8]:
def extract_json(text):

    matches = re.findall(
        r'\{.*?\}',
        text,
        re.DOTALL
    )

    for item in matches:

        try:
            return json.loads(item)

        except:
            pass

    return None

In [9]:
import re

def ocr_agent(document_text):

    prompt = f"""
Extract customer details from the document.

Document:

{document_text}

Return JSON only.
"""

    response = llm(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )[0]["generated_text"]

    print(response)

    # fallback extraction
    name = re.search(
        r"Name:\s*(.*)",
        document_text
    )

    dob = re.search(
        r"DOB:\s*(.*)",
        document_text
    )

    pan = re.search(
        r"PAN:\s*(.*)",
        document_text
    )

    return OCRResult(
        name=name.group(1).strip(),
        dob=dob.group(1).strip(),
        pan=pan.group(1).strip()
    )

In [11]:
document_text = ocr_documents_df.iloc[0][
    "document_text"
]

ocr_result = ocr_agent(
    document_text
)

print(ocr_result)


Extract customer details from the document.

Document:


Name: Aarav Mehta
DOB: 12/06/1994
PAN: ABCDE1234F


Return JSON only.
{"customer_details": {"name": "Aarav Mehta", "dob": "12/06/1994", "pan": "ABCDE1234F"}}
You are correct. The provided JSON accurately extracts the customer details from the given document. Here is the confirmation:

```json
{
  "customer_details": {
    "name": "Aarav Mehta",
    "dob": "12/06/1994",
   
name='Aarav Mehta' dob='12/06/1994' pan='ABCDE1234F'


In [12]:
from difflib import SequenceMatcher

def identity_agent(
    ocr_result,
    customer_record
):

    name_score = SequenceMatcher(
        None,
        ocr_result.name.lower(),
        customer_record["name"].lower()
    ).ratio()

    dob_match = (
        ocr_result.dob ==
        customer_record["dob"]
    )

    pan_match = (
        ocr_result.pan ==
        customer_record["pan"]
    )

    final_score = (
        0.4 * name_score +
        0.3 * int(dob_match) +
        0.3 * int(pan_match)
    )

    prompt = f"""
You are a KYC analyst.

Name Similarity Score:
{name_score}

DOB Match:
{dob_match}

PAN Match:
{pan_match}

Explain the result in one sentence.
"""

    response = llm(
        prompt,
        max_new_tokens=80
    )[0]["generated_text"]

    return IdentityResult(
        match_score=round(final_score,2),
        match_status=(
            "MATCH"
            if final_score > 0.9
            else "MISMATCH"
        ),
        reasoning=f"""
Name Similarity: {round(name_score,2)}
DOB Match: {dob_match}
PAN Match: {pan_match}
"""
    )

In [13]:
customer_record = customer_master_df.iloc[0]

identity_result = identity_agent(
    ocr_result,
    customer_record
)

print(identity_result)

match_score=1.0 match_status='MATCH' reasoning='\nName Similarity: 1.0\nDOB Match: True\nPAN Match: True\n'


In [14]:
transactions_df = pd.DataFrame([

    {
        "customer_id":1,
        "amount":50000,
        "type":"credit",
        "mode":"salary"
    },

    {
        "customer_id":1,
        "amount":25000,
        "type":"debit",
        "mode":"cash"
    },

    {
        "customer_id":1,
        "amount":15000,
        "type":"debit",
        "mode":"cash"
    },

    {
        "customer_id":1,
        "amount":12000,
        "type":"debit",
        "mode":"upi"
    },

    {
        "customer_id":1,
        "amount":100000,
        "type":"credit",
        "mode":"bonus"
    }

])

transactions_df

,customer_id,amount,type,mode
0,1,50000,credit,salary
1,1,25000,debit,cash
2,1,15000,debit,cash
3,1,12000,debit,upi
4,1,100000,credit,bonus


In [15]:
def transaction_feature_agent(txns):

    total_credit = sum(
        x["amount"]
        for x in txns
        if x["type"]=="credit"
    )

    total_debit = sum(
        x["amount"]
        for x in txns
        if x["type"]=="debit"
    )

    cash_txns = [
        x for x in txns
        if x["mode"]=="cash"
    ]

    cash_ratio = (
        len(cash_txns)
        / len(txns)
    )

    avg_transaction = (
        sum(x["amount"] for x in txns)
        / len(txns)
    )

    high_value_txn_count = len([
        x for x in txns
        if x["amount"] > 50000
    ])

    return TransactionFeatures(
        total_credit=total_credit,
        total_debit=total_debit,
        cash_ratio=round(cash_ratio,2),
        avg_transaction=round(avg_transaction,2),
        high_value_txn_count=high_value_txn_count
    )

In [16]:
txns = transactions_df.to_dict(
    orient="records"
)

features = transaction_feature_agent(
    txns
)

print(features)

total_credit=150000.0 total_debit=52000.0 cash_ratio=0.4 avg_transaction=40400.0 high_value_txn_count=1


In [17]:
def financial_profile_agent(features):

    prompt = f"""
You are a senior banking analyst.

Customer Features:

Total Credit: {features.total_credit}
Total Debit: {features.total_debit}
Cash Ratio: {features.cash_ratio}
Average Transaction: {features.avg_transaction}
High Value Transactions: {features.high_value_txn_count}

Provide a short customer profile and risk observations.
"""

    response = llm(
        prompt,
        max_new_tokens=150
    )[0]["generated_text"]

    profile = "Salary Based Customer"

    risks = []

    if features.cash_ratio > 0.5:
        risks.append("High Cash Usage")

    if features.high_value_txn_count > 0:
        risks.append("High Value Transactions")

    if features.total_credit > 100000:
        risks.append("High Income Customer")

    return {
        "profile": profile,
        "risk_indicators": risks,
        "ai_analysis": response[-500:]
    }

In [18]:
profile_result = financial_profile_agent(
    features
)

print(profile_result)

{'profile': 'Salary Based Customer', 'risk_indicators': ['High Value Transactions', 'High Income Customer'], 'ai_analysis': "al credit limit of $150,000 and a debit balance of $52,000, the customer has a relatively high utilization rate (34.67%) on their credit line. This suggests they may be close to their borrowing limit.\n- **Financial Health:** The cash ratio of 0.4 indicates that the customer's current assets are only about 40% of their current liabilities, which might suggest some financial vulnerability in case of an unexpected financial setback.\n- **Transaction Behavior:** The average transaction size of $40,40"}


In [19]:
def risk_agent(
    identity_result,
    features,
    profile_result
):

    risk_score = 0

    # Identity Risk

    if identity_result.match_score < 0.80:
        risk_score += 40

    # Cash Usage

    if features.cash_ratio > 0.50:
        risk_score += 25

    # High Value Transactions

    if features.high_value_txn_count > 0:
        risk_score += 15

    # Very High Credits

    if features.total_credit > 500000:
        risk_score += 10

    # Risk Level

    if risk_score >= 60:
        risk_level = "HIGH"

    elif risk_score >= 30:
        risk_level = "MEDIUM"

    else:
        risk_level = "LOW"

    prompt = f"""
You are a Senior KYC Risk Officer.

Customer Identity Result:

{identity_result}

Customer Transaction Features:

{features}

Customer Financial Profile:

{profile_result}

Risk Score:
{risk_score}

Risk Level:
{risk_level}

Explain why the customer received this risk level in 3 lines.
"""

    response = llm(
        prompt,
        max_new_tokens=120
    )[0]["generated_text"]

    return RiskResult(
        risk_score=risk_score,
        risk_level=risk_level,
        explanation=response[-500:]
    )

In [21]:
risk_result = risk_agent(
    identity_result,
    features,
    profile_result
)

print(risk_result)

risk_score=15.0 risk_level='LOW' explanation='core:\n15\n\nRisk Level:\nLOW\n\nExplain why the customer received this risk level in 3 lines.\nThe customer received a LOW risk level primarily due to matching their PAN and DOB, indicating a valid identity. However, given their high-value transactions and high income profile, coupled with a relatively high credit utilization, there is a potential for higher risk scenarios. Despite these factors, the overall low match score and transaction behavior indicate a lower risk compared to other profiles. ```'


In [22]:
def human_review_agent(risk_result):

    if risk_result.risk_level == "HIGH":

        return HumanReviewResult(
            escalate=True,
            review_queue="MANUAL_REVIEW"
        )

    return HumanReviewResult(
        escalate=False,
        review_queue="AUTO_APPROVED"
    )

In [23]:
review_result = human_review_agent(
    risk_result
)

print(review_result)

escalate=False review_queue='AUTO_APPROVED'


In [24]:
!pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 16.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 90.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [langgraph]14 [langgraph]sdk]]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [25]:
from typing import TypedDict

class KYCState(TypedDict):

    customer_id:int

    document_text:str

    ocr_result:dict

    identity_result:dict

    transaction_features:dict

    profile_result:dict

    risk_result:dict

    review_result:dict

In [26]:
def ocr_node(state):

    result = ocr_agent(
        state["document_text"]
    )

    return {
        "ocr_result":
        result.model_dump()
    }


def identity_node(state):

    customer_record = customer_master_df[
    customer_master_df["customer_id"]
    ==
    state["customer_id"]
].iloc[0]

    result = identity_agent(
        OCRResult(
            **state["ocr_result"]
        ),
        customer_record
    )

    return {
        "identity_result":
        result.model_dump()
    }


def transaction_node(state):

    txns = transactions_df[
    transactions_df["customer_id"]
    ==
    state["customer_id"]
].to_dict(
    orient="records"
)

    result = transaction_feature_agent(
        txns
    )

    return {
        "transaction_features":
        result.model_dump()
    }


def profile_node(state):

    result = financial_profile_agent(
        TransactionFeatures(
            **state[
                "transaction_features"
            ]
        )
    )

    return {
        "profile_result":
        result
    }


def risk_node(state):

    result = risk_agent(
        IdentityResult(
            **state[
                "identity_result"
            ]
        ),

        TransactionFeatures(
            **state[
                "transaction_features"
            ]
        ),

        state[
            "profile_result"
        ]
    )

    return {
        "risk_result":
        result.model_dump()
    }


def review_node(state):

    result = human_review_agent(
        RiskResult(
            **state[
                "risk_result"
            ]
        )
    )

    return {
        "review_result":
        result.model_dump()
    }

In [27]:
from langgraph.graph import (
    StateGraph,
    END
)

workflow = StateGraph(
    KYCState
)

workflow.add_node(
    "ocr",
    ocr_node
)

workflow.add_node(
    "identity",
    identity_node
)

workflow.add_node(
    "transactions",
    transaction_node
)

workflow.add_node(
    "profile",
    profile_node
)

workflow.add_node(
    "risk",
    risk_node
)

workflow.add_node(
    "review",
    review_node
)

In [28]:
workflow.set_entry_point(
    "ocr"
)

workflow.add_edge(
    "ocr",
    "identity"
)

workflow.add_edge(
    "identity",
    "transactions"
)

workflow.add_edge(
    "transactions",
    "profile"
)

workflow.add_edge(
    "profile",
    "risk"
)

workflow.add_edge(
    "risk",
    "review"
)

workflow.add_edge(
    "review",
    END
)

In [29]:
graph = workflow.compile()

print("Graph Compiled")

Graph Compiled


In [30]:
document_text = (
    ocr_documents_df
    .iloc[0]
    ["document_text"]
)

result = graph.invoke({

    "customer_id":1,

    "document_text":
    document_text

})
result


Extract customer details from the document.

Document:


Name: Aarav Mehta
DOB: 12/06/1994
PAN: ABCDE1234F


Return JSON only.
{"customer_details": {"name": "Aarav Mehta", "dob": "12/06/1994", "pan": "ABCDE1234F"}}
You are correct. The provided JSON accurately extracts the customer details from the given document. Here is the confirmation:

```json
{
  "customer_details": {
    "name": "Aarav Mehta",
    "dob": "12/06/1994",
   


{'customer_id': 1,
 'document_text': '\nName: Aarav Mehta\nDOB: 12/06/1994\nPAN: ABCDE1234F\n',
 'ocr_result': {'name': 'Aarav Mehta',
  'dob': '12/06/1994',
  'pan': 'ABCDE1234F'},
 'identity_result': {'match_score': 1.0,
  'match_status': 'MATCH',
  'reasoning': '\nName Similarity: 1.0\nDOB Match: True\nPAN Match: True\n'},
 'transaction_features': {'total_credit': 150000.0,
  'total_debit': 52000.0,
  'cash_ratio': 0.4,
  'avg_transaction': 40400.0,
  'high_value_txn_count': 1},
 'profile_result': {'profile': 'Salary Based Customer',
  'risk_indicators': ['High Value Transactions', 'High Income Customer'],
  'ai_analysis': "tern. The average transaction size is relatively high at $40,400, suggesting frequent large-scale purchases or investments. The presence of only one high-value transaction might imply that this customer is cautious about making large expenditures, possibly due to recent losses or a need for stability in their financial position.\n\nRisk Observations:\n- Given the